# DSFB SRD Figure Regeneration

This notebook reads CSV outputs from `output-dsfb-srd/<timestamp>/` and regenerates the three paper-facing figures for the deterministic Structural Regime Dynamics demonstrator.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

RUN_NAME = None  # Example: "20260312-214530"
REPO_ROOT = None  # Optional: Path("/content/dsfb")
OUTPUT_ROOT = None  # Optional: Path("/content/output-dsfb-srd")

def discover_output_root():
    candidates = []

    if OUTPUT_ROOT is not None:
        candidates.append(Path(OUTPUT_ROOT).expanduser())

    if REPO_ROOT is not None:
        candidates.append(Path(REPO_ROOT).expanduser() / "output-dsfb-srd")

    cwd = Path.cwd().resolve()
    for base in [cwd, *cwd.parents]:
        candidates.append(base / "output-dsfb-srd")

    candidates.extend(
        [
            Path("/content/dsfb/output-dsfb-srd"),
            Path("/content/output-dsfb-srd"),
            Path("/content/drive/MyDrive/dsfb/output-dsfb-srd"),
            Path("/content/drive/MyDrive/output-dsfb-srd"),
        ]
    )

    unique_candidates = []
    seen = set()
    for candidate in candidates:
        candidate = candidate.expanduser()
        if candidate not in seen:
            seen.add(candidate)
            unique_candidates.append(candidate)

    for candidate in unique_candidates:
        if candidate.exists():
            return candidate

    searched = "\n".join(f"- {candidate}" for candidate in unique_candidates)
    raise FileNotFoundError(
        "Could not locate output-dsfb-srd.\n"
        "Set OUTPUT_ROOT directly, or clone/copy the repo or output folder into Colab first.\n"
        "Searched:\n"
        f"{searched}"
    )

OUTPUT_ROOT = discover_output_root()
available_runs = sorted(path for path in OUTPUT_ROOT.iterdir() if path.is_dir())
if not available_runs:
    raise FileNotFoundError(f"No timestamped runs found under {OUTPUT_ROOT}")

RUN_DIR = OUTPUT_ROOT / RUN_NAME if RUN_NAME else available_runs[-1]
if not RUN_DIR.exists():
    raise FileNotFoundError(f"Requested run directory not found: {RUN_DIR}")

print(f"Using output root: {OUTPUT_ROOT}")
print(f"Using run directory: {RUN_DIR}")

In [ ]:
manifest = pd.read_csv(RUN_DIR / "run_manifest.csv")
events = pd.read_csv(RUN_DIR / "events.csv")
threshold_sweep = pd.read_csv(RUN_DIR / "threshold_sweep.csv")
transition_sharpness = pd.read_csv(RUN_DIR / "transition_sharpness.csv")
time_local = pd.read_csv(RUN_DIR / "time_local_metrics.csv")

manifest

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in threshold_sweep.groupby("n_events"):
    subset = subset.sort_values("tau_threshold")
    ax.plot(
        subset["tau_threshold"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 1: Connectivity Collapse $\\rho(\\tau)$")
ax.set_xlabel("Trust threshold $\\tau$")
ax.set_ylabel("Reachable fraction $\\rho(\\tau)$")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
for n_events, subset in transition_sharpness.groupby("n_events"):
    subset = subset.sort_values("tau_midpoint")
    ax.plot(
        subset["tau_midpoint"],
        subset["abs_drho_dtau"],
        linewidth=2,
        label=f"N={n_events}",
    )

ax.set_title("Figure 2: Transition Sharpness $|d\\rho / d\\tau|$")
ax.set_xlabel("Threshold midpoint")
ax.set_ylabel("Absolute derivative")
ax.legend()
fig.tight_layout()
plt.show()

In [ ]:
manifest_row = manifest.iloc[0]
shock_start = manifest_row["shock_start"]
shock_end = manifest_row["shock_end"]

time_local = time_local.copy()
time_local["window_midpoint"] = 0.5 * (time_local["window_start"] + time_local["window_end"])

fig, ax = plt.subplots(figsize=(10, 6))
for tau_threshold, subset in time_local.groupby("tau_threshold"):
    subset = subset.sort_values("window_midpoint")
    ax.plot(
        subset["window_midpoint"],
        subset["reachable_fraction"],
        linewidth=2,
        label=f"tau={tau_threshold:.2f}",
    )

ax.axvspan(shock_start, shock_end, color="tomato", alpha=0.18, label="shock interval")
ax.set_title("Figure 3: Time-Local Connectivity During Structural Regimes")
ax.set_xlabel("Event index")
ax.set_ylabel("Window reachable fraction")
ax.legend()
fig.tight_layout()
plt.show()